In [50]:
#add libraries
%pip install pandas matplotlib numpy
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Note: you may need to restart the kernel to use updated packages.


### Read Data

In [51]:
def read_data():
    return [
        pd.read_csv('../data/development_data/awards_players.csv'),
        pd.read_csv('../data/development_data/coaches.csv'),
        pd.read_csv('../data/development_data/players.csv'),
        pd.read_csv('../data/development_data/players_teams.csv'),
        pd.read_csv('../data/development_data/series_post.csv'),
        pd.read_csv('../data/development_data/teams.csv'),
        pd.read_csv('../data/development_data/teams_post.csv')
    ]

awards_players, coaches, players, players_teams, series_post, teams, teams_post = read_data()

### Data Selection

In [52]:
awards_players = awards_players.drop(columns=['lgID']) 
coaches = coaches.drop(columns=['lgID'])
players = players.drop(columns=[ 'college', 'collegeOther', 'deathDate'])
players_teams = players_teams.drop(columns=['lgID'])
series_post = series_post.drop(columns=['lgIDWinner', 'lgIDLoser']) #'round', 'series'??
teams = teams.drop(columns=['lgID', 'franchID', 'confID', 'divID', 'arena', 'name'])
teams_post = teams_post.drop(columns=['lgID'])



### Data Merging

In [53]:
#team metrics

data = pd.merge(teams, teams_post, on=['year', 'tmID'], how='left')

data['playoff_qualification'] = data['playoff'].apply(lambda x: 1 if x == 'Y' else 0)
data.drop(columns=['playoff'], inplace=True)
data.fillna({'W' : 0, 'L' : 0}, inplace=True)
    
#player stats
player_stats = players_teams.groupby(['tmID', 'year']).agg({
    'points': 'sum',
    'rebounds': 'sum',
    'assists': 'sum',
    'steals': 'sum',
    'blocks': 'sum',
    'turnovers': 'sum'
}).reset_index()

data = pd.merge(data, player_stats, on=['year', 'tmID'], how='left')

coach_stats = coaches.groupby(['year', 'tmID']).agg({
    'won': 'sum',
    'lost': 'sum',
    'post_wins': 'sum',
    'post_losses': 'sum'
}).reset_index()

data = pd.merge(data, coach_stats, on=["year", "tmID"], how="left")


data.columns

#awards
# awards_count = awards_players.groupby(['playerID', 'year']).size().reset_index(name='num_awards')

# data = pd.merge(data, awards_count, on=["year", "playerID"], how="left")
# data['num_awards'] = data['num_awards'].fillna(0)

Index(['year', 'tmID', 'rank', 'seeded', 'firstRound', 'semis', 'finals',
       'o_fgm', 'o_fga', 'o_ftm', 'o_fta', 'o_3pm', 'o_3pa', 'o_oreb',
       'o_dreb', 'o_reb', 'o_asts', 'o_pf', 'o_stl', 'o_to', 'o_blk', 'o_pts',
       'd_fgm', 'd_fga', 'd_ftm', 'd_fta', 'd_3pm', 'd_3pa', 'd_oreb',
       'd_dreb', 'd_reb', 'd_asts', 'd_pf', 'd_stl', 'd_to', 'd_blk', 'd_pts',
       'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB', 'won_x',
       'lost_x', 'GP', 'homeW', 'homeL', 'awayW', 'awayL', 'confW', 'confL',
       'min', 'attend', 'W', 'L', 'playoff_qualification', 'points',
       'rebounds', 'assists', 'steals', 'blocks', 'turnovers', 'won_y',
       'lost_y', 'post_wins', 'post_losses'],
      dtype='object')

In [54]:
data['win_loss_ratio'] = data['won_x'] / (data['won_x'] + data['lost_x'])  
data['avg_points_per_player'] = data['points'] / data['rank'] 

columns_to_drop = ['tmID', 'lgID', 'rank', 'firstRound', 'semis', 'finals']
data = data.drop(columns=columns_to_drop, errors='ignore')

In [55]:
data.fillna(0, inplace=True)

### Model training

In [56]:
%pip install scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

Note: you may need to restart the kernel to use updated packages.


#### Initialization 

In [57]:
#result lists
accuracy_scores = []
f1_scores = []
precision_scores = []
recall_scores = []

#drop useless columns
#data.drop(columns=['lgID_x', 'franchID'], inplace=True)

#separate feature and target columns
feature_columns = [col for col in data.columns if col not in ['playoff_qualification']]
target_column = 'playoff_qualification'

#Create model
model = RandomForestClassifier(random_state=21)
#model = LogisticRegression(random_state=21)
#model = DecisionTreeClassifier(random_state=21)

#### Year training cycle

In [59]:
for year in sorted(data['year'].unique())[:-1]:  # Leave out the last year since it won't have a year after it for testing
    # Separate train and test data
    train_data = data[data['year'] == year]
    test_data = data[data['year'] == year + 1]
    
    # If there's no data for the next year (e.g., last year in the dataset), skip
    if test_data.empty:
        continue

    # Split features and target
    X_train = train_data[feature_columns]
    y_train = train_data[target_column]
    X_test = test_data[feature_columns]
    y_test = test_data[target_column]
    
    # Standardize the data
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Train the model
    model.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred = model.predict(X_test)

    # Evaluate performance
    accuracy_scores.append(accuracy_score(y_test, y_pred))
    f1_scores.append(f1_score(y_test, y_pred))
    precision_scores.append(precision_score(y_test, y_pred))
    recall_scores.append(recall_score(y_test, y_pred))

    # Output results for each year
    print(f"Year {year} -> {year + 1}:")
    print(f"  Accuracy: {accuracy_scores[-1]:.2f}")
    print(f"  F1 Score: {f1_scores[-1]:.2f}")
    print(f"  Precision: {precision_scores[-1]:.2f}")
    print(f"  Recall: {recall_scores[-1]:.2f}")

Year 1 -> 2:
  Accuracy: 1.00
  F1 Score: 1.00
  Precision: 1.00
  Recall: 1.00
Year 2 -> 3:
  Accuracy: 0.94
  F1 Score: 0.94
  Precision: 0.89
  Recall: 1.00
Year 3 -> 4:
  Accuracy: 1.00
  F1 Score: 1.00
  Precision: 1.00
  Recall: 1.00
Year 4 -> 5:
  Accuracy: 1.00
  F1 Score: 1.00
  Precision: 1.00
  Recall: 1.00
Year 5 -> 6:
  Accuracy: 1.00
  F1 Score: 1.00
  Precision: 1.00
  Recall: 1.00
Year 6 -> 7:
  Accuracy: 0.93
  F1 Score: 0.94
  Precision: 0.89
  Recall: 1.00
Year 7 -> 8:
  Accuracy: 1.00
  F1 Score: 1.00
  Precision: 1.00
  Recall: 1.00
Year 8 -> 9:
  Accuracy: 0.93
  F1 Score: 0.94
  Precision: 0.89
  Recall: 1.00
Year 9 -> 10:
  Accuracy: 0.85
  F1 Score: 0.86
  Precision: 1.00
  Recall: 0.75


### End Results

In [60]:
print(f"Accura{accuracy_scores}")
print("\nAverage Performance Over All Years:")
print(f"  Average Accuracy: {sum(accuracy_scores) / len(accuracy_scores):.2f}")
print(f"  Average F1 Score: {sum(f1_scores) / len(f1_scores):.2f}")
print(f"  Average Precision: {sum(precision_scores) / len(precision_scores):.2f}")
print(f"  Average Recall: {sum(recall_scores) / len(recall_scores):.2f}")

Accura[1.0, 0.9375, 1.0, 1.0, 1.0, 0.9285714285714286, 1.0, 0.9285714285714286, 0.8461538461538461]

Average Performance Over All Years:
  Average Accuracy: 0.96
  Average F1 Score: 0.96
  Average Precision: 0.96
  Average Recall: 0.97
